# 01 — Canonical Verdicts: Cross-family ingestion

**Abstract.** This notebook is the **reproducibility companion** to the
stochastic-FX variance-proxy working paper at
`docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex`. Three
trios will progressively render the paper's §7 results:

- **Trio 1 (this dispatch)** — load and cross-check the three
  authoritative canonical-pin `InversionVerdict` JSON sidecars (GBM, OU,
  Merton) emitted by the Task-5 IO Boundary classes; reproduce paper
  §7.1's cross-family summary table.
- Trio 2 — per-family interpretation figures (paper §7.2).
- Trio 3 — strip-preservation + determinism + Wave-2 summary (paper
  §7.3–§7.5).

The notebook is **verify-only** at runtime: it reads the committed
JSON sidecars under `simulations/stochastic_fx/data/` via the
`InversionVerdictEmitter().read(family_id)` round-trip API and does
**not** re-run the Phase B Monte-Carlo or Phase C KS dispatch. Re-running
those would touch the source-of-truth verdicts and is out of scope.

**Reproducibility pin.** All numerics here are anchored to
`CANONICAL_RNG_SEED = 42` (the paper §7 anchor) — see
`simulations/notebooks/stochastic_fx/env.py`.

**Decision-citation block — canonical-pin verdict source**

- **Reference**: `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex` §1 (introduction) + §7.1 (cross-family results table); `docs/specs/2026-05-11-stochastic-fx-variant-design.md` v0.5 §4.2 (canonical pins) and §5 (Pin Z1.x anti-fishing invariants).
- **Why used**: The committed per-family `inversion_verdict_{family_id}.json` sidecars under `simulations/stochastic_fx/data/` are the authoritative single-source-of-truth for the canonical-pin verification verdicts. Spec v0.5 §5 Pin Z1.5 explicitly bans recomputation-from-source in any downstream consumer — the JSON sidecars are the audit-anchored artefact, and only an `InversionVerdictEmitter().read(...)` round-trip is permitted (which re-applies the frozen-dc invariants in `InversionVerdict.__post_init__`).
- **Relevance to results**: If this decision were different — e.g., if the notebook re-ran the Phase B Monte-Carlo locally — the per-family rel-errs and KS p-values would drift across executions (Phase C has finite-sample noise), the paper §7.1 cross-family table would no longer be byte-stable, and Pin Z1.5 anti-fishing would be violated. Reading the sidecars via the emitter API instead preserves the audit-block hex prefixes (paper §7.1 column 7) and gives Type-I-rejection nullity at fixed `seed = 42`.
- **Connection to simulator**: Maps to `simulations/stochastic_fx/emit.py::InversionVerdictEmitter` (IO Boundary tier) — the only sanctioned reader of the JSON sidecars; its `read(family_id)` method validates `schema_version == 'v1.0'` and reconstructs the frozen-dc `InversionVerdict` via `simulations/stochastic_fx/types.py::InversionVerdict`.

In [ ]:
from __future__ import annotations

import pandas as pd

from simulations.notebooks.stochastic_fx.env import (
    CANONICAL_RNG_SEED,
    DATA_ROOT,
    reproducibility_seed,
)
from simulations.stochastic_fx import (
    InversionVerdictEmitter,
    KS_PVALUE_FLOOR,
    MOMENT_REL_TOL,
    NUMERICAL_IDENTITY_TOL,
)

_seed = reproducibility_seed()
print(f"canonical rng seed: {_seed} (env.CANONICAL_RNG_SEED = {CANONICAL_RNG_SEED})")
print(f"data root         : {DATA_ROOT}")
print(
    f"tolerances        : phase_a = {NUMERICAL_IDENTITY_TOL}, "
    f"phase_b = {MOMENT_REL_TOL}, phase_c = {KS_PVALUE_FLOOR}"
)

## Trio 1 — Cross-family verdict ingestion

**Why this trio.** Paper §7.1 reports a single cross-family summary table
consolidating the three canonical-pin verification verdicts (GBM, OU,
Merton). The table is the entry point to §7.2's per-family interpretation
and is the visible artefact a reader of the paper checks first. This
trio loads the three `InversionVerdict` JSON sidecars — the authoritative
canonical-pin verdict source per spec v0.5 §5 (Pin Z1.x anti-fishing) —
and reconstructs that table from a pandas DataFrame indexed by family.

The sidecars are read via
`InversionVerdictEmitter().read(family_id)` rather than `json.load`
directly: the emitter API re-applies the schema-v1.0 invariants (missing
keys → `RoundTripDriftError`; bad schema_version → `RoundTripDriftError`)
and runs the frozen-dc `InversionVerdict.__post_init__` invariants
(composite-AND check on the three phase-pass flags, finite-float checks
on the residuals / rel-errs / p-value). Any drift in the on-disk JSON
fails fast here rather than silently producing a stale table.

**Reference**: paper `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex` §7.1 (table layout); `notes/STOCHASTIC_FX_RESULTS.md` §1 (same table rendered as markdown); spec v0.5 §6 (three-phase verification semantics).

In [ ]:
# Load the three canonical-pin verdicts via the sanctioned IO Boundary
# reader. The emitter is configured with the project DATA_ROOT (not the
# emitter's default relative path) so this notebook is location-agnostic.
emitter = InversionVerdictEmitter(emit_dir=DATA_ROOT)

FAMILY_IDS: tuple[str, ...] = ("gbm", "ou", "merton")
FAMILY_LABELS: dict[str, str] = {
    "gbm": "GBM",
    "ou": "OU",
    "merton": "Merton",
}

verdicts = {fid: emitter.read(fid) for fid in FAMILY_IDS}

# Build the paper §7.1 cross-family summary DataFrame.
rows: list[dict[str, object]] = []
for fid in FAMILY_IDS:
    v = verdicts[fid]
    rows.append(
        {
            "family": FAMILY_LABELS[fid],
            "composite_pass": v.composite_pass,
            "phase_a_max_residual": v.phase_a_max_residual,
            "phase_b_mean_rel_err": v.phase_b_mean_rel_err,
            "phase_b_var_rel_err": v.phase_b_var_rel_err,
            "phase_c_ks_pvalue": v.phase_c_ks_pvalue,
            "audit_block_prefix": v.audit_block[:16],
        }
    )

df = pd.DataFrame(rows).set_index("family")

# Headless-stable rendering — print the full string repr.
print(df.to_string())

**Interpretation — per-family numerics at `seed = 42`.**

Reading column by column against the paper §7.1 thresholds:

- **`composite_pass`** — all three families PASS, matching paper §7.1
  row-1 verdicts. The composite-AND invariant (`phase_a_pass AND
  phase_b_pass AND phase_c_pass`) is enforced in
  `InversionVerdict.__post_init__`, so a `True` here is
  emitter-validated.
- **`phase_a_max_residual`** — algebraic-identity residuals are at
  machine epsilon for all three families (GBM ≈ 7e-15, OU ≈ 4e-15,
  Merton ≈ 2e-13), all far below `NUMERICAL_IDENTITY_TOL = 1e-06`.
  Merton is two orders larger than GBM/OU because the compound-Poisson
  jump term contributes additional float-arithmetic accumulations;
  still 7 orders below the gate.
- **`phase_b_mean_rel_err`** — Phase B mean-only gate per spec v0.5
  §11.6 (Pin Z1.3b). Observed: GBM ≈ 1.12 %, OU ≈ 0.17 %, Merton ≈
  0.70 %. All three are comfortably below `MOMENT_REL_TOL = 0.05`
  (5 %); the OU value is smallest because the Vasicek-style exact
  transition density carries the lowest MC-sampling noise on the σ_T
  mean. This is the gate that drives `phase_b_pass`.
- **`phase_b_var_rel_err`** — paper §11.6 audit-trail observation
  only: NOT a gate. Observed: GBM ≈ 0.213, OU ≈ 0.027, Merton ≈ 2.73.
  Merton's value is roughly 100× OU's because Merton's high jump
  kurtosis inflates the finite-sample variance estimator at `N_PATHS =
  1000`; the disposition under Pin Z1.3b is to surface this number
  rather than gate on it (gating would force a non-canonical
  variance-shrinkage on the jump-diffusion family — anti-fishing per
  spec v0.5 §11.6).
- **`phase_c_ks_pvalue`** — Phase C KS goodness-of-fit floor per spec
  v0.5 §11.7 (Pin Z1.4). Observed: GBM ≈ 0.123, OU ≈ 0.303, Merton ≈
  0.116. All three are comfortably above `KS_PVALUE_FLOOR = 0.01`, so
  there is **no Type-I rejection at `seed = 42`** on any family.
  Merton uses the empirical-CDF reference dispatch per Pin Z1.4 (high-N
  reference run at `N_REF = 100_000`, `N_REF_SEED = 20260513`); GBM and
  OU use their closed-form CDFs.
- **`audit_block_prefix`** — the first 16 hex chars of the
  SHA-256 audit block bound to `(canonical_params, rng_seed,
  schema_version)`. These prefixes are paper §7.1 column 7 and are
  the per-family lineage anchors.

**Reference**: paper `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex` §7.1 (cross-family table values), §11.6 (Phase B variance rel-err audit-trail disposition), §11.7 (Phase C per-family reference dispatch); spec v0.5 §11.6–§11.7 (Pin Z1.3b and Pin Z1.4).

---

**HALT — end of Trio 1.** Per `memory/feedback_notebook_trio_checkpoint.md`,
the next trio (per-family interpretation figures) is a separate dispatch
after human review of this DataFrame.

**Decision-citation block — per-family sample-path visualization (Trio 2)**

- **Reference**: `docs/papers/2026-05-16-stochastic-fx-variance-proxy-paper.tex`
  §7.2 (Per-family interpretation), per-family paragraphs *GBM — lognormal
  absorbs first two moments*, *OU — Gaussian quadratic form, gamma reference*,
  *Merton — mixture geometry, empirical-CDF dispatch*; spec
  `docs/specs/2026-05-11-stochastic-fx-variant-design.md` v0.3 §4.2 canonical
  numerical pins; `simulations/stochastic_fx/types.py` ``CANONICAL_GBM``,
  ``CANONICAL_OU``, ``CANONICAL_MERTON``.
- **Why**: paper §7.2 narrates per-family qualitative character (GBM
  log-multiplicative dispersion, OU stationary mean-reversion to $\bar\mu$,
  Merton heavy-tail Poisson-mixture-of-lognormals) but reports only summary
  scalar verdicts in §7.1. A small-ensemble sample-path figure is the natural
  visual anchor for those narratives — it lets the reader see the SDE-family
  signature without re-running the 1000-path canonical ensemble. The trio
  re-instantiates the three generators directly from the verified
  ``simulations.stochastic_fx`` package at ``CANONICAL_RNG_SEED = 42``, so the
  figure is reproducible bit-exactly to the same RNG stream consumed by the
  paper §7 audit_blocks.
- **Relevance**: the figure feeds paper §7.2 as
  ``\includegraphics{per_family_sample_paths.pdf}``. Renders 20 sample paths
  per family (vector PDF), publication-quality serif typography, Type-42 font
  embedding. Empirical-visual only: no analytic moment overlay (those are
  Trio 3's responsibility — σ_T accumulator trace + cross-family ratio).
- **Connection**: identical ``(params, rng_seed, n_paths)`` reproducibility
  contract as `simulations.stochastic_fx.generators` (Pin Z1.2). The 20-path
  sub-ensemble is independent of the 1000-path verified ensemble at
  ``N_REF_SEED`` — it is a visualization-only draw at the canonical pin seed
  ``42``, n_paths=20.


In [ ]:
# Per-family sample-path visualization — paper §7.2 figure.
#
# Re-instantiates the three PathGenerators at the canonical pin and draws
# n_paths = 20 trajectories per family at CANONICAL_RNG_SEED = 42. Renders
# a 3-panel landscape figure (one panel per family) and persists both a
# vector PDF (for paper inclusion) and a PNG (for in-notebook display).

from __future__ import annotations

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

from simulations.notebooks.stochastic_fx.env import (
    CANONICAL_RNG_SEED,
    FIGURES_DIR,
)
from simulations.stochastic_fx import (
    CANONICAL_GBM,
    CANONICAL_MERTON,
    CANONICAL_OU,
    GBMPathGenerator,
    JumpDiffusionPathGenerator,
    OUPathGenerator,
)

# Publication-quality typography (econ-visualization conventions).
mpl.rcParams["font.family"] = "serif"
mpl.rcParams["font.size"] = 10
mpl.rcParams["pdf.fonttype"] = 42  # TrueType embedding for vector PDF.
mpl.rcParams["ps.fonttype"] = 42

N_PATHS_FIG: int = 20
# Plot-grid decimation stride: the underlying MC ensemble retains its full
# bit-exact 5001-step reproducibility (paths.shape unchanged), but the
# vector PDF only renders every PLOT_STRIDE-th grid point. At dt = T/5000
# ≈ 2.4e-3 yr, stride=25 yields ~200 points per path — visually identical
# to the full-resolution curve at 4-inch panel width while keeping the PDF
# below the 500 KB paper-inclusion limit.
PLOT_STRIDE: int = 25

# Re-instantiate the three generators at the canonical pin and draw the
# small visualization ensemble. The same (params, rng_seed, n_paths)
# tuple is reproducible bit-exactly per the Pin Z1.2 contract.
gbm_gen = GBMPathGenerator(params=CANONICAL_GBM)
ou_gen = OUPathGenerator(params=CANONICAL_OU)
merton_gen = JumpDiffusionPathGenerator(params=CANONICAL_MERTON)

ensembles = {
    "GBM": (gbm_gen(rng_seed=CANONICAL_RNG_SEED, n_paths=N_PATHS_FIG), CANONICAL_GBM),
    "OU": (ou_gen(rng_seed=CANONICAL_RNG_SEED, n_paths=N_PATHS_FIG), CANONICAL_OU),
    "Merton": (
        merton_gen(rng_seed=CANONICAL_RNG_SEED, n_paths=N_PATHS_FIG),
        CANONICAL_MERTON,
    ),
}

# Build per-family title strings citing the canonical-pin parameters.
def _gbm_title(p) -> str:  # noqa: ANN001
    return f"GBM: $x_0={p.x_0:.0f}$, $\\sigma={p.sigma:.4f}$, $T={p.T:.0f}$"

def _ou_title(p) -> str:  # noqa: ANN001
    return (
        f"OU: $x_0={p.x_0:.0f}$, $\\bar\\mu={p.mu_bar:.0f}$, "
        f"$\\theta={p.theta:.1f}$, $T={p.T:.0f}$"
    )

def _merton_title(p) -> str:  # noqa: ANN001
    return (
        f"Merton: $x_0={p.x_0:.0f}$, $\\sigma={p.sigma:.4f}$, "
        f"$\\lambda={p.lambda_jump:.1f}$, $T={p.T:.0f}$"
    )

titles = {
    "GBM": _gbm_title(ensembles["GBM"][1]),
    "OU": _ou_title(ensembles["OU"][1]),
    "Merton": _merton_title(ensembles["Merton"][1]),
}

fig, axes = plt.subplots(1, 3, figsize=(12.0, 4.0), sharey=False)

for ax, family in zip(axes, ("GBM", "OU", "Merton")):
    ensemble, params = ensembles[family]
    paths_full = ensemble.paths
    n_paths, n_grid = paths_full.shape
    t_grid_full = np.linspace(0.0, params.T, n_grid)

    # Decimate plot grid only (ensemble itself untouched).
    plot_idx = np.concatenate(
        [np.arange(0, n_grid, PLOT_STRIDE), np.array([n_grid - 1])]
    )
    plot_idx = np.unique(plot_idx)
    t_grid = t_grid_full[plot_idx]
    paths = paths_full[:, plot_idx]

    # Light grey individual paths.
    for k in range(n_paths):
        ax.plot(t_grid, paths[k], color="0.55", alpha=0.4, linewidth=0.7)

    # Ensemble mean (solid blue) — computed on the FULL ensemble, then
    # decimated for plotting, to preserve the moment-statistic accuracy.
    mean_path = paths_full.mean(axis=0)[plot_idx]
    ax.plot(t_grid, mean_path, color="C0", linewidth=1.6, label="ensemble mean")

    # 95% ensemble band (dashed black) — same pattern: percentile on full
    # ensemble, decimate for plotting.
    lower = np.percentile(paths_full, 5, axis=0)[plot_idx]
    upper = np.percentile(paths_full, 95, axis=0)[plot_idx]
    ax.plot(t_grid, lower, color="black", linestyle="--", linewidth=1.0, label="5/95 pct")
    ax.plot(t_grid, upper, color="black", linestyle="--", linewidth=1.0)

    ax.set_title(titles[family])
    ax.set_xlabel("time $t$ (years)")
    ax.set_ylabel("FX rate $X_t$")
    ax.legend(loc="best", frameon=False, fontsize=8)
    ax.grid(True, alpha=0.25)

fig.tight_layout()

# Persist publication-quality artefacts. The PDF is the load-bearing
# artefact (paper §7.2 inclusion); the PNG is a convenience render for
# notebook display.
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
pdf_path = FIGURES_DIR / "per_family_sample_paths.pdf"
png_path = FIGURES_DIR / "per_family_sample_paths.png"
fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
fig.savefig(png_path, format="png", dpi=200, bbox_inches="tight")
plt.show()
print(f"wrote {pdf_path.relative_to(pdf_path.parents[3])}")
print(f"wrote {png_path.relative_to(png_path.parents[3])}")
print(f"PDF size: {pdf_path.stat().st_size / 1024:.1f} KB")


**Interpretation — per-family sample-path signatures.**

The three panels render the qualitative SDE-family character that paper §7.2
*Per-family interpretation* describes in prose. Each panel shows 20 light-grey
trajectories at the canonical pin, the ensemble-mean path (solid blue), and
the 5th/95th percentile envelope (dashed black) at every grid point on
$t \in [0, T=12]$.

**GBM panel — log-multiplicative drift, $\sqrt{t}$-widening band.** With
canonical pin $\mu = 0$, $\sigma = 0.10/\sqrt{12} \approx 0.0289$, the
$\sigma^{2} T = 0.01$ regime produces modest dispersion: the 5/95 envelope
fans out monotonically as approximately $\sqrt{t}$ around the (zero-drift)
mean, and the ensemble mean tracks $x_{0} = 4000$ to within MC-sampling noise.
This is the log-multiplicative tail that the paper §7.2 *GBM* paragraph
attributes to the bivariate-lognormal-quadratic-form true distribution; the
moderate KS $p$-value ($0.123$) reflects the higher-order shape discrepancy
not visible at this small-ensemble scale.

**OU panel — stationary mean-reversion to $\bar\mu$.** With $\theta = 1.0$
and $\bar\mu = x_{0} = 4000$, the mean-reversion timescale is $1/\theta = 1$
year; the 5/95 envelope reaches its stationary width within roughly three
timescales and remains stable thereafter — there is no long-term drift, only
stationary excursions around $\bar\mu$. This is the joint-Gaussian quadratic
form regime that yields the cleanest Phase B (mean rel-err $1.68 \times
10^{-3}$) and Phase C ($p = 0.303$) verdicts of the three families.

**Merton panel — heavy-tail Poisson jumps over GBM-like diffusion.** With
$\lambda = 1.0$ jump per year, most paths track a GBM-like continuous
diffusion punctuated by occasional discrete jumps; the 5/95 envelope is
visibly wider than GBM's at every $t > 0$ from the compound-Poisson
contribution. This is the Poisson-mixture-of-lognormals geometry that paper
§7.2 *Merton* identifies as the reason joint Gaussianity fails and the
audit-trail variance rel-err inflates to $2.73$ — corrected by the
empirical-CDF Phase C dispatch (spec v0.5 §11.7).

**HALT — end of Trio 2**
